# Create Fine-Tuning JSONL Data

This notebook converts the 30% NYC training subset produced by the training-subset notebook into JSONL files for the XCrime-LLM fine-tuning workflow.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

### Build Fine-Tuning JSONL Files

Load the 30% NYC training subset and full validation split, construct the compact XCrime-LLM prompts and labels, and save the resulting datasets in chat-format JSONL for fine-tuning.

### Load and Prepare Training and Validation Data

In [ ]:
import json
import os

# Paths produced by the previous notebooks
DATA_DIR = "/content/drive/MyDrive/XCrime-LLM/data/splits"

TRAIN_SUB_PATH = f"{DATA_DIR}/master_train_30pct.csv"
VAL_PATH = f"{DATA_DIR}/master_val.csv"

OUTPUT_DIR = "/content/drive/MyDrive/XCrime-LLM/data/fine_tuning"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def detect_date_col(df: pd.DataFrame) -> str:
    if "date" in df.columns:
        return "date"
    if "complaint_dt" in df.columns:
        return "complaint_dt"
    raise KeyError("Expected a 'date' or 'complaint_dt' column.")


def apply_guardrails(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    x = df.copy()

    x[date_col] = pd.to_datetime(
        x[date_col],
        errors="coerce",
    )

    x["region_id"] = pd.to_numeric(
        x.get("region_id"),
        errors="coerce",
    ).astype("Int64")

    x["crime"] = (
        x.get("crime", "")
        .astype("string")
        .str.upper()
        .str.strip()
    )

    for col in [
        "last7_total",
        "last28_mean",
        "R1_influence",
        "base_rate",
    ]:
        if col in x.columns:
            x[col] = (
                pd.to_numeric(x[col], errors="coerce")
                .fillna(0.0)
                .astype(float)
            )

    if "base_rate" in x.columns:
        x["base_rate"] = x["base_rate"].clip(0.0, 1.0)

    if "recency" in x.columns:
        x["recency"] = pd.to_numeric(
            x["recency"],
            errors="coerce",
        )

        x["recency"] = np.where(
            x["recency"].isna(),
            9999,
            x["recency"],
        )

        x["recency"] = np.where(
            x["recency"] == 9999,
            9999,
            np.minimum(x["recency"], 365),
        ).astype(int)

    for col in ["dow", "month"]:
        if col in x.columns:
            x[col] = (
                pd.to_numeric(x[col], errors="coerce")
                .fillna(0)
                .astype(int)
            )

    return x


EVENT_MAPPING = {
    "BURGLARY": "EVENT_TYPE_A",
    "ROBBERY": "EVENT_TYPE_B",
    "GRAND LARCENY": "EVENT_TYPE_C",
    "FELONY ASSAULT": "EVENT_TYPE_D",
}


def align_long(
    df: pd.DataFrame,
    date_col: str,
) -> pd.DataFrame:
    x = df.copy()

    if "event_type" not in x.columns:
        x["event_type"] = x["crime"].map(EVENT_MAPPING)

    x["date"] = pd.to_datetime(
        x[date_col],
        errors="coerce",
    )

    keep = [
        "region_id",
        "date",
        "crime",
        "event_type",
        "last7_total",
        "last28_mean",
        "recency",
        "R1_influence",
        "base_rate",
        "dow",
        "month",
        "label_7d",
    ]

    x = x[
        [col for col in keep if col in x.columns]
    ].copy()

    anchor_counts = (
        x.groupby(["region_id", "date"])["event_type"]
        .nunique()
    )

    assert (
        anchor_counts == 4
    ).all(), "Some (region_id, date) anchors are missing event rows."

    return x


# Load the 30% training subset
train_sub = pd.read_csv(TRAIN_SUB_PATH)

train_date_col = detect_date_col(train_sub)
train_sub = apply_guardrails(
    train_sub,
    train_date_col,
)

train_long_30 = align_long(
    train_sub,
    train_date_col,
)

print(
    "[OK] TRAIN anchors:",
    train_long_30.groupby(
        ["region_id", "date"]
    ).ngroups,
)

# Load the full validation split
val_df = pd.read_csv(VAL_PATH)

val_date_col = detect_date_col(val_df)
val_df = apply_guardrails(
    val_df,
    val_date_col,
)

val_long = align_long(
    val_df,
    val_date_col,
)

print(
    "[OK] VAL anchors:",
    val_long.groupby(
        ["region_id", "date"]
    ).ngroups,
)

### Define the Fine-Tuning Prompt

In [ ]:
EVENT_ORDER = [
    "EVENT_TYPE_A",
    "EVENT_TYPE_B",
    "EVENT_TYPE_C",
    "EVENT_TYPE_D",
]


def fmt_int(value):
    try:
        return str(int(value))
    except (TypeError, ValueError):
        return "0"


def fmt_float(value, decimals=4):
    try:
        value = float(value)
        if np.isfinite(value):
            return f"{value:.{decimals}f}"
    except (TypeError, ValueError):
        pass

    return f"{0.0:.{decimals}f}"


def make_schema_json(events):
    return (
        "{"
        + ",".join(
            [f'"{event}":0' for event in events]
        )
        + "}"
    )


def build_system_msg(events=EVENT_ORDER) -> str:
    event_list = ", ".join(events)
    schema_json = make_schema_json(events)

    return (
        "You are a spatiotemporal analyst.\n"
        f"For the SAME (REGION, DATE), output independent 0/1 forecasts for each of: {event_list} "
        "for whether ≥1 incident will occur in the NEXT 7 DAYS.\n"
        "Rules: decide independently; use ONLY provided numeric features; no outside knowledge; "
        "output JSON ONLY (no prose).\n"
        "Feature notes: last7_total↑, last28_mean↑, base_rate↑, R1_influence↑ ⇒ higher risk; "
        "recency lower ⇒ higher risk (9999=never). dow=0–6, month=1–12.\n"
        "Return ONE compact JSON object with exactly these keys and 0/1 values:\n"
        f"{schema_json}\n"
    )


def build_prompt_for_anchor(
    group: pd.DataFrame,
) -> str:
    region_id = int(group["region_id"].iloc[0])

    date = pd.to_datetime(
        group["date"].iloc[0],
        errors="coerce",
    )

    date_str = (
        str(date.date())
        if pd.notna(date)
        else ""
    )

    dow = (
        int(date.weekday())
        if pd.notna(date)
        else -1
    )

    month = (
        int(date.month)
        if pd.notna(date)
        else 0
    )

    lines = [
        f"RID={region_id};DT={date_str};D={dow};M={month}"
    ]

    for event_type in EVENT_ORDER:
        row = group.loc[
            group["event_type"] == event_type
        ].head(1)

        if len(row):
            record = row.iloc[0]

            parts = [
                f"last7_total={fmt_int(record.get('last7_total', 0))}",
                f"recency={fmt_int(record.get('recency', 9999))}",
                f"base_rate={fmt_float(record.get('base_rate', 0.0), 4)}",
                f"R1_influence={fmt_float(record.get('R1_influence', 0.0), 3)}",
                f"last28_mean={fmt_float(record.get('last28_mean', 0.0), 3)}",
            ]

        else:
            parts = [
                "last7_total=0",
                "recency=9999",
                "base_rate=0.0000",
                "R1_influence=0.000",
                "last28_mean=0.000",
            ]

        lines.append(
            f"{event_type}: " + " ".join(parts)
        )

    return "\n".join(lines)

### Generate Fine-Tuning JSONL Files

In [ ]:
def build_prompts_df(
    long_df: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for (region_id, date), group in long_df.groupby(
        ["region_id", "date"],
        sort=False,
    ):
        rows.append(
            {
                "region_id": int(region_id),
                "date": pd.to_datetime(date),
                "multi_prompt": build_prompt_for_anchor(group),
            }
        )

    return pd.DataFrame(rows)


def labels_wide_from_long(
    long_df: pd.DataFrame,
) -> pd.DataFrame:
    return (
        long_df
        .pivot_table(
            index=["region_id", "date"],
            columns="event_type",
            values="label_7d",
            aggfunc="max",
        )
        .reindex(columns=EVENT_ORDER)
        .fillna(0)
        .astype(int)
        .reset_index()
    )


train_prompts = build_prompts_df(train_long_30)
val_prompts = build_prompts_df(val_long)

train_labels = labels_wide_from_long(
    train_long_30
)
val_labels = labels_wide_from_long(
    val_long
)

train_join = train_prompts.merge(
    train_labels,
    on=["region_id", "date"],
    validate="one_to_one",
)

val_join = val_prompts.merge(
    val_labels,
    on=["region_id", "date"],
    validate="one_to_one",
)

print(
    "[OK] TRAIN lines:",
    len(train_join),
    "| VAL lines:",
    len(val_join),
)

SYSTEM_MSG = build_system_msg(EVENT_ORDER)


def write_jsonl_chat(
    df: pd.DataFrame,
    path: str,
):
    with open(
        path,
        "w",
        encoding="utf-8",
    ) as file:
        for _, row in df.iterrows():
            target = {
                event: int(row[event])
                for event in EVENT_ORDER
            }

            record = {
                "messages": [
                    {
                        "role": "system",
                        "content": SYSTEM_MSG,
                    },
                    {
                        "role": "user",
                        "content": row["multi_prompt"],
                    },
                    {
                        "role": "assistant",
                        "content": json.dumps(
                            target,
                            ensure_ascii=False,
                            separators=(",", ":"),
                        ),
                    },
                ]
            }

            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    print(
        f"[SAVE] {path} "
        f"(lines={len(df)})"
    )


ft_train_jsonl = (
    f"{OUTPUT_DIR}/ft_train_30pct_LT.jsonl"
)

ft_val_jsonl = (
    f"{OUTPUT_DIR}/ft_val_LT.jsonl"
)

write_jsonl_chat(
    train_join,
    ft_train_jsonl,
)

write_jsonl_chat(
    val_join,
    ft_val_jsonl,
)

# Preview the first record from each file
for path in [
    ft_train_jsonl,
    ft_val_jsonl,
]:
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        first_record = next(file).strip()

    print(
        f"[HEAD] {path} → "
        f"{first_record[:220]}…"
    )